In [0]:
# #import files from silver layer
# from pyspark.sql.functions import col, timestamp, lit
# from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType

#load match results data
df_silver_match_results = spark.read.table("rugby_data_dev.rugby_silver.match_results_notebook")




In [0]:
#import libaries and functions 
from pyspark.sql import SparkSession
from pyspark.sql.functions import lit
import pandas as pd

spark = SparkSession.builder.appName("Rugby Elo Ratings").getOrCreate()

#create elo system
#-=================
#Base Elo is hard set to 1500
#K factor is hard set to 35
#-=================


#create dictonaries
eloRatings = {}
history = {}

#get Elo ratings: if the team is not in the eloRatings dictionary, add it with a rating of 1500
def getElo(team):
    return eloRatings.get(team, 1500)

#update the eloRating based on the match result
def updateElo(home, away, result):
    homeElo = getElo(home)
    awayElo = getElo(away)

    #set K factor 
    K = 35 
    
    #convert match result to numeric value from the home prospective
    if result.lower() == 'home win':
        matchResult = 1
    elif result.lower() == 'away win':
        matchResult = 0
    else:
        matchResult = 0.5

    #calculate the expected result from the home prospective
    expectedResult = 1 / (1 + 10 ** ((awayElo - homeElo) / 400))

    #update elo ratings and store
    newHomeElo = homeElo + K * (matchResult - expectedResult)
    newAwayElo = awayElo + K * ((1 - matchResult) - (1 - expectedResult))

    eloRatings[home] = newHomeElo
    eloRatings[away] = newAwayElo

    #tracks the rating change in history
    for team, rating in [(home, newHomeElo), (away, newAwayElo)]:
        history.setdefault(team, []).append(rating)

    return homeElo, awayElo, newHomeElo, newAwayElo

#get history function
def getHistory():
    return history

# =================
#Process the silver table 
# ================

#convert to Pandas: -> this is needed as PySpark runs in Parrell and Elo is dependant on the previous value
matches = (
    df_silver_match_results
    .select('MatchId', 'Season', 'Round', 'HomeTeam', 'AwayTeam', 'Result')
    .orderBy('Season', 'Round')
    .toPandas()
)


# =============
#Compute the Elo sequentaily
# =============

results = []

for _, row in matches.iterrows():
    homeTeam = row['HomeTeam']
    awayTeam = row['AwayTeam']
    result = row['Result']
    
    homeEloBefore, awayEloBefore, homeEloAfter, awayEloAfter, = updateElo(homeTeam, awayTeam, result)
    
    results.append({
        'MatchId': row['MatchId'],
        'Season': row['Season'],
        'Round': row['Round'],
        'HomeTeam': homeTeam,
        'AwayTeam': awayTeam,
        'Result': result,
        'HomeEloBefore': homeEloBefore,
        'AwayEloBefore': awayEloBefore,
        'HomeEloAfter': homeEloAfter,
        'AwayEloAfter': awayEloAfter,
        'HomeEloChange': homeEloAfter - homeEloBefore,
        'AwayEloChange': awayEloAfter - awayEloBefore,
    })

#round results to 1 decimal place
results_df = pd.DataFrame(results).round({
    'HomeEloBefore': 1,
    'AwayEloBefore': 1,
    'HomeEloAfter': 1,
    'AwayEloAfter': 1,
    'HomeEloChange': 1,
    'AwayEloChange': 1
})


# ============== 
# Convert back to Spark + Write to Gold table
# ==============

df_elo = spark.createDataFrame(pd.DataFrame(results_df))

df_elo = (
    df_elo
    .withColumn('pipelineStage', lit('goldTransformation'))
)

df_elo.write.format('delta').mode('overwrite').option('overwriteSchema', 'true').saveAsTable('rugby_data_dev.rugby_gold.elo_ratings')

df_elo.show(200)